# Lab Main Pipeline

Clean mainframe notebook with three current workflow routes.


## 1. Load Package

Run this every session. It is fast and does not ask for the ZIP unless `lab_pipeline` is missing.


In [ ]:
from pathlib import Path
import os
import sys

NOTEBOOK_NAMES = [
    'Lab_App_Colab_UI.ipynb',
    "Lab_Main_Pipeline.ipynb",
    "Generar_Estructura.ipynb",
    "Lab_Group_Manager.ipynb",
    "Import_Teammates_to_Notas.ipynb",
]


def running_in_colab():
    try:
        import google.colab
        return True
    except Exception:
        return False


def mount_drive_if_colab():
    if running_in_colab():
        from google.colab import drive
        drive.mount("/content/drive")


def find_notebook_workspace():
    if not running_in_colab():
        return Path.cwd()

    mydrive = Path("/content/drive/MyDrive")
    matches = []
    for notebook_name in NOTEBOOK_NAMES:
        matches.extend(mydrive.rglob(notebook_name))

    if matches:
        newest = max(matches, key=lambda path: path.stat().st_mtime)
        return newest.parent

    workspace = mydrive / "Generador_GNT"
    workspace.mkdir(parents=True, exist_ok=True)
    return workspace


def force_workspace_first(workspace_dir):
    workspace_text = str(workspace_dir)
    sys.path[:] = [p for p in sys.path if p != workspace_text]
    sys.path.insert(0, workspace_text)


def clear_lab_pipeline_modules():
    for module_name in list(sys.modules):
        if module_name == "lab_pipeline" or module_name.startswith("lab_pipeline."):
            del sys.modules[module_name]


def load_lab_pipeline():
    mount_drive_if_colab()
    workspace_dir = find_notebook_workspace()
    os.environ["LAB_PIPELINE_WORKSPACE_DIR"] = str(workspace_dir)
    force_workspace_first(workspace_dir)

    if not (workspace_dir / "lab_pipeline").exists():
        raise ModuleNotFoundError(
            "No encontre lab_pipeline en la carpeta del notebook. "
            "Ejecute la celda opcional 'Actualizar paquete' y suba el ZIP."
        )

    clear_lab_pipeline_modules()
    import lab_pipeline
    print(f"Carpeta base de trabajo: {workspace_dir}")
    print("lab_pipeline loaded.")
    print(f"lab_pipeline file: {Path(lab_pipeline.__file__).resolve()}")
    return workspace_dir


WORKSPACE_DIR = load_lab_pipeline()


## 2. Actualizar Paquete (Opcional)

Run this only when you have a new `lab_pipeline_package.zip`. Then rerun **Load Package**.


In [ ]:
from pathlib import Path
import zipfile

PACKAGE_ZIP = "lab_pipeline_package.zip"
PACKAGE_ZIP_PREFIX = "lab_pipeline_package"


def find_uploaded_package(uploaded):
    for uploaded_name in uploaded:
        path = Path(uploaded_name)
        if path.suffix.lower() == ".zip" and path.stem.startswith(PACKAGE_ZIP_PREFIX):
            return path
    raise FileNotFoundError("Debes subir lab_pipeline_package.zip.")


def update_lab_pipeline_package():
    if not running_in_colab():
        raise RuntimeError("Esta celda de actualizacion esta pensada para Colab.")

    from google.colab import files

    workspace_dir = find_notebook_workspace()
    print(f"Sube ahora {PACKAGE_ZIP}.")
    uploaded = files.upload()
    package_path = find_uploaded_package(uploaded)

    with zipfile.ZipFile(package_path, "r") as z:
        z.extractall(workspace_dir)

    print(f"Paquete instalado/actualizado en: {workspace_dir}")
    print("Ahora vuelva a correr la celda 1: Load Package.")


update_lab_pipeline_package()


## 3. Choose Route

Run this after **Load Package**.


In [ ]:
from lab_pipeline.config import LabConfig
from lab_pipeline.structure import run_structure_workflow
from lab_pipeline.lab_groups import run_lab_group_workflow
from lab_pipeline.teammates import run_teammates_workflow

config = LabConfig()

ROUTES = {
    "1": ("Generar estructura", lambda: run_structure_workflow()),
    "2": ("Crear Grupos de Laboratorio", lambda: run_lab_group_workflow(config)),
    "3": ("Subiendo notas", lambda: run_teammates_workflow(config)),
}

print("Que desea hacer?")
for key, (title, _) in ROUTES.items():
    print(f"  ({key}) {title}")

choice = input("Choose route 1, 2, or 3: ").strip()
if choice not in ROUTES:
    raise ValueError("Invalid route. Choose 1, 2, or 3.")

title, route_fn = ROUTES[choice]
print(f"Running: {title}")
result = route_fn()
